# Official crawl of the 20,000 IDs = M1 benchmark (Google Colab, CPU-only)

Owner: Hiếu (Member 3) · drafted by Thiên, 27/10/2026. Drafted with the help of an AI assistant (Claude, Anthropic), cited per the course rules.

The statement measures the running time over *each of the required papers* with *no upper limit on the running time*, on *a Google Colab instance running on CPU-only mode*. The official crawl is therefore **the benchmark**: every chunk of `CHUNK` IDs runs under `/usr/bin/time -v` and `src/utils/profiler.py` (resident memory of all processes, disk, output size) and writes `work/profile/<chunk>/`. `python -m src.analysis.stats` sums the chunks afterwards.

**Rules (team plan, sections 2.3 and 3.4):**

- Only **one** shift runs at a time, because arXiv allows one connection over all machines of the group. The notebook writes `SHIFT.json` on the shared Drive and **refuses to start** while another member's shift was active in the last 45 minutes.
- Corpus and bookkeeping (`data/`, `work/`) live on the shared Drive. A shift copies them to the Colab disk, runs, and syncs back after every chunk.
- Session lost: reopen the notebook and **Run all**; finished versions are skipped.
- End of shift: run the last cell to sync and release the shift, then report in the group chat.

Before running: Colab Secrets hold `S2_API_KEY` and `ARXIV_CONTACT_EMAIL`, and `DRIVE_DIR` is shared with the whole group. Runtime: *Runtime → Change runtime type → CPU*.

In [ ]:
# Shift parameters: edit this cell only
SHIFT_OWNER = "thien"  # who is on shift: thien | tien | hieu | dang
RANGES = ["thien", "tien", "hieu", "dang"]  # crawl order, one instance for the 4 ranges
RANGE_SIZE = 5000
CHUNK = 500  # sync to Drive and close one profile every CHUNK IDs
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"  # shared group folder
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
BRANCH = "main"
REFRESH_METADATA = False  # True for the FIRST shift of the official crawl only (plan, 1.3)
FORCE = False  # True only when you are sure the previous shift has stopped

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess

if os.path.isdir("/content/CodeRepo/.git"):
    subprocess.run(["git", "-C", "/content/CodeRepo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/CodeRepo"], check=True)
%cd /content/CodeRepo
# Install dependencies without breaking Google Colab's pre-installed packages
!pip install -q --no-deps -r src/requirements.txt || pip install -q -r src/requirements.txt
!test -x /usr/bin/time || (apt-get -qq update && apt-get -qq install -y time)
!git log --oneline -1

In [ ]:
import os

from google.colab import userdata

# Direct configuration fallback if not configured in Colab Secrets:
DIRECT_SECRETS = {
    "S2_API_KEY": "",  # Paste key here if not using Colab Secrets (empty = public 3s/req mode)
    "ARXIV_CONTACT_EMAIL": "",  # Paste contact email here if not using Colab Secrets
}

for name in ("S2_API_KEY", "ARXIV_CONTACT_EMAIL"):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = str(val).strip()
            print(f"{name}: set from Colab Secrets")
            continue
    except Exception:
        pass
    if DIRECT_SECRETS.get(name):
        os.environ[name] = DIRECT_SECRETS[name].strip()
        print(f"{name}: set from DIRECT_SECRETS fallback")
    else:
        print(f"{name}: NOT set (using default public mode)")

In [ ]:
# Machine configuration, reported in the book
!nproc
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h
!df -h /content
!python --version
!nvidia-smi -L || echo "No GPU: CPU-only runtime as required"

In [ ]:
# Shift marker on the shared Drive: two shifts never run at the same time
import json
import time
from datetime import UTC, datetime
from pathlib import Path

DRIVE = Path(DRIVE_DIR)
LOCAL = Path("/content/run")
MARKER = DRIVE / "SHIFT.json"
for d in (DRIVE / "data" / "corpus", DRIVE / "work", LOCAL / "data" / "corpus", LOCAL / "work"):
    d.mkdir(parents=True, exist_ok=True)


def heartbeat(released=False):
    now = datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
    MARKER.write_text(
        json.dumps({"owner": SHIFT_OWNER, "at": time.time(), "at_iso": now, "released": released})
    )


if MARKER.exists() and not FORCE:
    m = json.loads(MARKER.read_text())
    age_min = (time.time() - m["at"]) / 60
    if m["owner"] != SHIFT_OWNER and not m.get("released") and age_min < 45:
        raise RuntimeError(
            f"{m['owner']}'s shift was active {age_min:.0f} minutes ago. "
            "Wait until they run the 'End of shift' cell, or ask in the group chat."
        )
heartbeat()
print("Shift taken by", SHIFT_OWNER)

In [ ]:
# Copy corpus and bookkeeping from Drive to the Colab disk (faster than writing to Drive)
!rsync -a {DRIVE_DIR}/data/ /content/run/data/
!rsync -a --exclude tmp/ {DRIVE_DIR}/work/ /content/run/work/
!du -sh /content/run/data /content/run/work

In [ ]:
import sys


def sync_to_drive():
    subprocess.run(["rsync", "-a", f"{LOCAL}/data/", f"{DRIVE}/data/"], check=True)
    subprocess.run(
        ["rsync", "-a", "--exclude", "tmp/", f"{LOCAL}/work/", f"{DRIVE}/work/"], check=True
    )
    heartbeat()


def run_chunk(name, offset):
    # One profile folder per chunk run; a resumed chunk gets its own folder too.
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S")
    prof = LOCAL / "work" / "profile" / f"{name}-{offset:04d}-{stamp}"
    prof.mkdir(parents=True)
    cmd = ["/usr/bin/time", "-v", "-o", str(prof / "time_v.log"), sys.executable]
    cmd += ["-m", "src.utils.profiler", "--out", str(prof)]
    cmd += ["--watch", f"{LOCAL}/data", "--watch", f"{LOCAL}/work"]
    cmd += ["--output", f"{LOCAL}/data/corpus", "--", sys.executable, "-m", "src.run_pipeline"]
    cmd += ["--range", name, "--offset", str(offset), "--limit", str(CHUNK)]
    cmd += ["--data-root", f"{LOCAL}/data/corpus", "--work-dir", f"{LOCAL}/work", "--no-progress"]
    if REFRESH_METADATA:
        cmd.append("--refresh-metadata")
    return subprocess.run(cmd).returncode


for name in RANGES:
    for offset in range(0, RANGE_SIZE, CHUNK):
        print(f"=== {name} {offset}..{offset + CHUNK - 1}", flush=True)
        code = run_chunk(name, offset)
        sync_to_drive()
        summary = json.loads((LOCAL / "work" / "run_summary.json").read_text())
        print({k: summary[k] for k in ("papers_complete", "ids_requested", "failed_rows")})
        if code == 2:
            raise RuntimeError("work/ is held by another run (run.lock)")
        if code != 0:
            raise RuntimeError(f"run_pipeline stopped with code {code}: see {LOCAL}/work/run.log")
print("All four ranges done.")

## End of shift

Run this cell when your shift ends, even if the crawl is not finished: it syncs one last time and releases the shift for the next member.

In [ ]:
sync_to_drive()
heartbeat(released=True)
!tail -n 3 /content/run/work/run.log
print("Shift released. Post in the group chat: versions done, failed_rows, RAM/disk.")